In [1]:
import holoviews as hv; hv.extension('bokeh', 'plotly', logo=None)
import panel as pn;     pn.extension()

from panel.interact import interact

#import scipy.io
import numpy as np
import os

from PIL import Image, ImageOps

def raster(img):  return hv.Raster(img).opts(cmap="gray", xaxis=None, yaxis=None, frame_width=200, aspect='equal')
def show(img):    return pn.Row( raster(img) )

<div style="float:center;width:100%;text-align: center;"><strong style="height:60px;color:darkred;font-size:40px;">Eigenfaces</strong></div>

# 1. A Database of Faces

Consider a collection of faces:

In [2]:
def load_faces():
    # https://github.com/vutsalsinghal/EigenFace/tree/master/Dataset/
    # Construct a list of demeaned arrays of each of the faces
    dataset_path = 'faces/'
    imgs     = [Image.open( os.path.join('faces',f'f{n}.jpg') ) for n in range(17)]
    new_imgs = []
    for img in imgs:
        a = np.array( img.crop( (20, 50, 175, 200 )) )
        new_imgs.append( (a - np.mean(a))/np.std(a) )
    return new_imgs
def show_faces( imgs, n_per_row=4,m_per_col=4 ):
    n,m  = imgs[0].shape
    img  = np.empty( (m_per_col*n,n_per_row*m))

    count = 0
    for j in range(m_per_col):
        for k in range(n_per_row):
            img[j*n:(j+1)*n,k*m:(k+1)*m] = imgs[count]
            count += 1
    return pn.Row(raster(img).opts(frame_width=400))

imgs         = load_faces()
tst_img      = imgs[2]; del imgs[2]
height,width = imgs[0].shape
N_imgs       = len(imgs)

show_faces(imgs, 8,2)

Row
    [0] HoloViews(Raster, sizing_mode='fixed')

**The idea** is to convert each image into a vector (i.e., a feature,)<br>
$\qquad$ write these into a matrix as columns, and **use PCA.**

We will need routines to effectuate the conversions

## 1.1 Useful Functions

In [3]:
# Feature Space
def vec_from_img(img):       return np.reshape(img, -1)
def img_from_vec(vec, h):    return vec.reshape((h,-1))
def feature_matrix( imgs ) : return np.array([vec_from_img(img) for img in imgs]).T

Next, set up routines for PCA:

$\qquad\begin{aligned}
& A\ =\ U_r\ \Sigma_r\ V_r^T & \qquad \text{ reduced SVD decompostion }\\
& P\ =\ U_r\ \Sigma_r        & \qquad \text{ PCA feature matrix }
\end{aligned}$

The PCA components are the columns of $P$, i.e $p_i = \sigma_i u_i$,<br>
$\qquad$ where $u_i$ is the $\text{i}^{th}$ column of $U_r$, and $\sigma_i$ is the corresponding singular value.

For simplicity, we will drop the $r$ subscripts in the code

In [4]:
# PCA feature space
def pca_vec(          i, U, Sigma ): return U[:,i] * Sigma[i]
def img_from_pca_vec( i, U, Sigma, h ): return img_from_vec( pca_vec(i, U, Sigma), h)

The columns of $U_r$ form an orthonormal basis for the feature space $\mathscr{C}(A)$

We set up routines to obtain the coordinate vector of a feature vector in this basis,<br>
$\qquad$ and its inverse:

$\qquad\begin{aligned}
& a =               U_r x\quad & \Leftrightarrow\quad x = U^T_r a \\
& a = \Sigma_r^{-1} P_r x\quad & \Leftrightarrow\quad x = P^T_r a \\
\end{aligned}
$

In [5]:
def coordinate_vector_u( feature, U ):        return U.T @ feature
def coordinate_vector_p( feature, P, Sigma):
    xp = P.T @ feature
    for i in range(len(xp)): xp[i] /= Sigma[i]
    return xp

def feature_from_coordinate_vector_u( x, U):  return U @ x
def feature_from_coordinate_vector_p( x, U, Sigma):
    for i in range(len(x)): x[i] *= Sigma[i]
    xp = U @ x
    return xp

In [6]:
print("Checks" )
A = np.random.normal( size=(10,5) )
U,Sigma,Vt = np.linalg.svd( A, full_matrices = False )
print( "feature <-> coordinate_vector u: ",
        np.linalg.norm(feature_from_coordinate_vector_u(
                       coordinate_vector_u( A[:,1], U), U)
                       - A[:,1]))
print( "feature <-> coordinate_vector p: ",
        np.linalg.norm(feature_from_coordinate_vector_p(
                       coordinate_vector_p( A[:,1], U, Sigma), U, Sigma)
                       - A[:,1]))
print( "vec -> Image -> vec:             ",
       np.linalg.norm( vec_from_img(img_from_vec(A[:,0],5))-A[:,0]))

Checks
feature <-> coordinate_vector u:  1.879402968499888e-15
feature <-> coordinate_vector p:  1.879402968499888e-15
vec -> Image -> vec:              0.0


# 2 PCA of the Face Features

## 2.1 The Mean Face: an Average of All the Faces

In [7]:
mean_face = imgs[0]
for img in imgs[1:]: mean_face = np.add( mean_face, img)
mean_face /= N_imgs
show(mean_face)

Row
    [0] HoloViews(Raster, sizing_mode='fixed')

Let's subtract it out of our images! This should improve the PCA features.

In [8]:
imgs     = [img-mean_face for img in imgs]
tst_img -= mean_face

## 2.2 The PCA Features

The full set of PCA features $p_i$ form an **orthogonal basis** for $\mathscr{C}(A)$,<br>
$\qquad$ with $\Vert p_i \Vert = \sigma_i.$

Let's look at the images in this basis

In [9]:
A          = feature_matrix(imgs)
U,Sigma,Vt = np.linalg.svd( A, full_matrices = False )
pca_A      = A @ Vt.T   # i.e., pca_A = A V = U Σ  the matrix of PCA features

In [10]:
pca_faces = [img_from_pca_vec(i, U, Sigma, height) for i in range(N_imgs)]
show_faces(pca_faces, 8,2)

Row
    [0] HoloViews(Raster, sizing_mode='fixed')

Let's look at the pca_features in more detail

In [11]:
interact( lambda i: show(pca_faces[i]), i=(0,N_imgs-1 ))

Column
    [0] Column
        [0] IntSlider(end=15, name='i', value=7)
    [1] Row
        [0] Row
            [0] HoloViews(Raster, sizing_mode='fixed')

We can **recover specific faces** from the features, since each face is encoded by the coordinate vector $V^T_i$:

In [12]:
interact( lambda i: show( mean_face+img_from_vec(pca_A @ Vt[:,i], height )), i=(0,N_imgs-1) )

Column
    [0] Column
        [0] IntSlider(end=15, name='i', value=7)
    [1] Row
        [0] Row
            [0] HoloViews(Raster, sizing_mode='fixed')

Each face can be represented by a coordinate vector with respect to the PCA features:

In [13]:
interact( lambda i: hv.Spikes((list(range(N_imgs)),Vt[:,i]),"PCA Feature Index", "Coefficient").opts( title="Coordinate Vector for Image i", width=350), i=(0,N_imgs-1) )

Column(sizing_mode='fixed')
    [0] Column
        [0] IntSlider(end=15, name='i', value=7)
    [1] Row(sizing_mode='fixed')
        [0] HoloViews(Spikes, height=300, name='interactive00408', sizing_mode='fixed', width=350)

## 2.3 Dimensionality Reduction

The PCA features are ordered by their importance (variance explained).<br>
$\quad$ Do we need all of them to reconstruct a figure, or could we truncate the expanison in the PCA basis?<br>

Let us look at the variance explained as a function of the number of PCA features included:

In [14]:
var_explained = 100*np.cumsum( Sigma*Sigma ) / sum(Sigma*Sigma)
hv.Curve(Sigma, "index", "σ")*hv.Scatter(Sigma, "index", "σ")\
  .opts(size=4, tools=["hover"], title="Singular Values", show_grid=True)+\
hv.Curve(var_explained,"index", "% var explained")\
  .opts(show_grid=True,title="% Variage explained")*\
hv.Scatter(var_explained).opts(size=4, tools=['hover'])

:Layout
   .Overlay.I  :Overlay
      .Curve.I   :Curve   [index]   (σ)
      .Scatter.I :Scatter   [index]   (σ)
   .Overlay.II :Overlay
      .Curve.I   :Curve   [index]   (% var explained)
      .Scatter.I :Scatter   [x]   (y)

In [15]:
interact( lambda num_pca_features,i:
            pn.Row( raster( mean_face+img_from_vec(pca_A[:, 0:num_pca_features] @ Vt[0:num_pca_features,i], height))\
                    .opts(title="Reconstructed Image") +
                    raster(mean_face+imgs[i]).opts(title="Original Image")),
          num_pca_features = (1,N_imgs,1), i=(0,N_imgs,1) )

Column
    [0] Column
        [0] IntSlider(end=16, name='num_pca_features', start=1, value=8)
        [1] IntSlider(end=16, name='i', value=8)
    [1] Row
        [0] Row
            [0] HoloViews(Layout)

**Remark:** the results would be much improved given a larger collection of faces.

## Can We Reconstruct a Test Image?

Given the PCA basis, we need to project the test image onto the span of that basis,<br>
$\qquad$ i.e., onto $\mathscr{C}(A)$

Success will depend on whether the test image is well approximated by this projection.

In [16]:
proj_tst_img = pca_A @ (pca_A.T @ vec_from_img(tst_img))
pn.Row(raster(mean_face+tst_img).opts(title="Test Image"),
       raster( mean_face+img_from_vec( proj_tst_img, height)).opts(title="Projected Image"))

Row
    [0] HoloViews(Raster, sizing_mode='fixed')
    [1] HoloViews(Raster, sizing_mode='fixed')

**Remark:** more images would significantly increase $dim \mathscr{C}(A)$!

This does raise an interesting question: can we ascertain<br>
$\qquad$ whether a given image is or is not represented in the database?

# 3. Facial Recognition

Are any of the pca_features be used to distinguish between images?
Let's compare the coefficients of the coordinate vectors for each pair of images.

In [17]:
# Obtain the coordinate vectors of each image
img_coordinates     = [coordinate_vector_p( A[:,i], pca_A, Sigma) for i in range(N_imgs)]
tst_img_coordinates = coordinate_vector_p( vec_from_img(tst_img), pca_A, Sigma)

# Compare each of the coefficients for across all images
distances = { i:[] for i in range(N_imgs)}
for i in range(N_imgs):
    for j in range(N_imgs):
        diff = img_coordinates[i]-img_coordinates[j]
        for coord_index in range(N_imgs):
            distances[coord_index].append(abs(diff[coord_index]))
means     = []
stds      = []
for i in range(N_imgs):
    means.append( np.mean( distances[i] ))
    stds.append(  np.std(  distances[i] ))

In [18]:
pn.Row(hv.Dataset(
 {'Values': np.concatenate([distances[i] for i in range(N_imgs)]),
  'Entry': np.concatenate([np.repeat(f'{i}', len(distances[0])) for i in range(N_imgs)]) },
  ['Entry'], 'Values').to(hv.Violin, 'Entry', 'Values').opts(width=800, title="Distribution of Coordinate Vector Entries"))

Row
    [0] HoloViews(Violin, height=300, sizing_mode='fixed', width=800)

Again, this needs more data. For this small data set, the first three coordinate_entries exhibit the most spread.

The test image as well as images 2 and 3 show the same person.<br>
Let's look at the first three coordinate vectors

In [19]:
#%output backend="plotly" size=120
hv.extension("plotly", logo=False)
all_cs = img_coordinates[    0:3]
tst_cs = tst_img_coordinates[0:3]

h=\
hv.Scatter3D((all_cs[0],     all_cs[1],     all_cs[2]),      label="3 coordinates" ).opts(size=2)*\
hv.Scatter3D((tst_cs[0],     tst_cs[1],     tst_cs[2]),      label="Test Image").opts(size=3)*\
hv.Scatter3D((all_cs[0][1:3],all_cs[1][1:3],all_cs[2][1:3]), label="Test Person").opts(size=3)
pn.Row(h)

Row
    [0] HoloViews(Overlay)

This is inconclusive: we need more data...